# Classificação de Padrões Socioterritoriais em Imagens de Satélite (EuroSAT)
### Trabalho Final de Visão Computacional (Projeto DataLuta)
**INSTRUÇÃO:** Vá no menu superior do Colab em `Ambiente de execução` > `Alterar tipo de ambiente de execução` e selecione **GPU (T4)** para treinar rápido.

In [ ]:
# 1. Instalação de Dependências
!pip install scikit-learn seaborn matplotlib torch torchvision tqdm pandas


## 2. Definindo o Pipeline de Dados (Dataset)

In [ ]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
import numpy as np

# Estatísticas padrão (aproximadas ImageNet/RGB) para normalização
EUROSAT_MEAN = [0.485, 0.456, 0.406]
EUROSAT_STD = [0.229, 0.224, 0.225]
SEED = 42

class DatasetWrapper(torch.utils.data.Dataset):
    """Wrapper para aplicar transformações específicas em subsets (treino ou validação)."""
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

def get_dataloaders(data_dir='./data', batch_size=64, use_augmentation=False, use_sampler=False):
    """
    Carrega o dataset EuroSAT, aplica o split de 80/20 com seed=42 e retorna os Dataloaders.
    """
    # Transformações de Validação (Sem Augmentation)
    val_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=EUROSAT_MEAN, std=EUROSAT_STD)
    ])

    # Transformações de Treino com Augmentation de Satélite
    if use_augmentation:
        train_transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.RandomHorizontalFlip(p=0.5), # Válido para satélites
            transforms.RandomVerticalFlip(p=0.5),   # Válido para satélites
            transforms.RandomRotation(360),         # Satélites não têm orientação fixa
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=EUROSAT_MEAN, std=EUROSAT_STD)
        ])
    else:
        train_transform = val_transform

    # O torchvision.datasets.EuroSAT cuida do download do arquivo zip
    base_dataset = datasets.EuroSAT(root=data_dir, download=True)
    
    # Extrair labels para estratificação e sampler
    labels = [target for _, target in base_dataset.samples]
    
    # Split fixo em 80% treino e 20% validação
    train_idx, val_idx = train_test_split(
        np.arange(len(base_dataset)),
        test_size=0.2,
        random_state=SEED,
        stratify=labels
    )
    
    # Criar subsets
    train_subset = Subset(base_dataset, train_idx)
    val_subset = Subset(base_dataset, val_idx)
    
    # Aplicar transforms específicos
    train_dataset = DatasetWrapper(train_subset, transform=train_transform)
    val_dataset = DatasetWrapper(val_subset, transform=val_transform)

    # Configuração do Sampler para rebalanceamento
    if use_sampler:
        train_labels = [labels[i] for i in train_idx]
        class_sample_count = np.array([len(np.where(train_labels == t)[0]) for t in np.unique(train_labels)])
        weight = 1. / class_sample_count
        samples_weight = np.array([weight[t] for t in train_labels])
        samples_weight = torch.from_numpy(samples_weight)
        sampler = WeightedRandomSampler(samples_weight.type('torch.DoubleTensor'), len(samples_weight))
        train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=2)
    else:
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, base_dataset.classes



## 3. Definindo as Arquiteturas (Modelos)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BaselineCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(BaselineCNN, self).__init__()
        # Extração direta de características locais
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=0)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=0)
        
        # Cálculo do tamanho do flatten para imagem 64x64:
        # 64x64 -> conv1(3x3) -> 62x62 -> pool2d -> 31x31
        # 31x31 -> conv2(3x3) -> 29x29 -> pool2d -> 14x14
        self.fc1 = nn.Linear(64 * 14 * 14, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class SmallVGG(nn.Module):
    def __init__(self, num_classes=10):
        super(SmallVGG, self).__init__()
        # Bloco 1 (32 -> 32)
        self.conv1_1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1_1 = nn.BatchNorm2d(32)
        self.conv1_2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn1_2 = nn.BatchNorm2d(32)
        
        # Bloco 2 (32 -> 64)
        self.conv2_1 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2_1 = nn.BatchNorm2d(64)
        self.conv2_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2_2 = nn.BatchNorm2d(64)
        
        # Bloco 3 (64 -> 128)
        self.conv3_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3_1 = nn.BatchNorm2d(128)
        self.conv3_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn3_2 = nn.BatchNorm2d(128)
        
        self.pool = nn.MaxPool2d(2, 2)
        
        # Classificador com Dropout de 30%
        # 64x64 -> pool1(32x32) -> pool2(16x16) -> pool3(8x8)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = F.relu(self.bn1_1(self.conv1_1(x)))
        x = F.relu(self.bn1_2(self.conv1_2(x)))
        x = self.pool(x)
        
        x = F.relu(self.bn2_1(self.conv2_1(x)))
        x = F.relu(self.bn2_2(self.conv2_2(x)))
        x = self.pool(x)
        
        x = F.relu(self.bn3_1(self.conv3_1(x)))
        x = F.relu(self.bn3_2(self.conv3_2(x)))
        x = self.pool(x)
        
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class SmallResNet(nn.Module):
    def __init__(self, num_classes=10):
        super(SmallResNet, self).__init__()
        self.in_channels = 32
        
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        
        self.layer1 = self._make_layer(32, stride=1)
        self.layer2 = self._make_layer(64, stride=2)
        self.layer3 = self._make_layer(128, stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.linear = nn.Linear(128, num_classes)

    def _make_layer(self, out_channels, stride):
        layer = ResidualBlock(self.in_channels, out_channels, stride)
        self.in_channels = out_channels
        return layer

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.linear(x)
        return x



## 4. Definindo Funções de Treino e Avaliação

In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm
import time

def train_model(model, train_loader, val_loader, optimizer_type='adam', epochs=15, lr=0.001, device=None):
    """
    Treina o modelo e registra o histórico de loss e acurácia.
    
    Args:
        model: Arquitetura a ser treinada.
        train_loader: DataLoader de treinamento.
        val_loader: DataLoader de validação.
        optimizer_type: 'sgd', 'momentum' ou 'adam'.
        epochs: Número de épocas de treinamento.
        lr: Learning rate.
        device: 'cuda' ou 'cpu'.
        
    Returns:
        history: Dicionário contendo loss e acurácia do treino e validação por época.
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    # Configuração do otimizador baseada na Seção 3 do projeto
    if optimizer_type == 'sgd':
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    elif optimizer_type == 'momentum':
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif optimizer_type == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Otimizador desconhecido: {optimizer_type}")

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    print(f"Iniciando treinamento com otimizador {optimizer_type.upper()} por {epochs} épocas no device {device}...")
    start_time = time.time()
    
    for epoch in range(epochs):
        # Fase de Treinamento
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        # tqdm para progress bar interativo no terminal/notebook
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            pbar.set_postfix({'loss': loss.item()})
            
        epoch_train_loss = running_loss / total_train
        epoch_train_acc = correct_train / total_train
        
        # Fase de Validação
        model.eval()
        running_val_loss = 0.0
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                running_val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                
        epoch_val_loss = running_val_loss / total_val
        epoch_val_acc = correct_val / total_val
        
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        
        print(f"Epoch {epoch+1}/{epochs} -> Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} "
              f"|| Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
              
    elapsed = time.time() - start_time
    print(f"Treinamento concluído em {elapsed // 60:.0f}m {elapsed % 60:.0f}s.")
    
    return history


import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

def evaluate_model(model, dataloader, class_names, device=None):
    """
    Avalia o modelo gerando métricas avançadas (Acurácia Macro, Precisão, Revocação, F1-Score).
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
    model = model.to(device)
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    # Relatório do Scikit-Learn
    print("=== Relatório de Classificação (Resultados) ===")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    
    macro_acc = accuracy_score(all_labels, all_preds)
    print(f"Acurácia Global (Micro-Averaged): {macro_acc:.4f}")
    
    return all_labels, all_preds

def plot_training_curves(history, title_suffix=""):
    """
    Plota as curvas de Loss e Acurácia de treinamento e validação.
    """
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gráfico de Loss
    ax1.plot(epochs, history['train_loss'], 'b-', label='Treino Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', label='Validação Loss')
    ax1.set_title(f'Função de Perda (Loss) - {title_suffix}')
    ax1.set_xlabel('Épocas')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    
    # Gráfico de Acurácia
    ax2.plot(epochs, history['train_acc'], 'b-', label='Treino Acurácia')
    ax2.plot(epochs, history['val_acc'], 'r-', label='Validação Acurácia')
    ax2.set_title(f'Acurácia - {title_suffix}')
    ax2.set_xlabel('Épocas')
    ax2.set_ylabel('Acurácia')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(f"{title_suffix.replace(' ', '_').replace('+', 'and')}_curves.png")
    plt.show()

def plot_confusion_matrix(y_true, y_pred, class_names):
    """
    Renderiza a matriz de confusão elegante usando Seaborn.
    """
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Matriz de Confusão do Modelo Vencedor (SmallResNet)')
    plt.ylabel('Classe Real')
    plt.xlabel('Classe Predita')
    plt.tight_layout()
    plt.savefig('confusion_matrix_resnet.png')
    plt.show()



## 5. Execução Oficial dos Experimentos

In [ ]:
print('=== Inicializando Experimentos no Colab ===')
train_loader, val_loader, class_names = get_dataloaders(data_dir='./data', use_augmentation=True, batch_size=128)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Rodando na placa de vídeo: {device}')

# Treino da Baseline
model_baseline = BaselineCNN(num_classes=10)
history_base = train_model(model_baseline, train_loader, val_loader, optimizer_type='adam', epochs=15, device=device)
plot_training_curves(history_base, 'Baseline CNN')

# Treino da ResNet (Campeã)
model_resnet = SmallResNet(num_classes=10)
history_res = train_model(model_resnet, train_loader, val_loader, optimizer_type='adam', epochs=15, device=device)
plot_training_curves(history_res, 'SmallResNet')

# Matriz de Confusão
y_true, y_pred = evaluate_model(model_resnet, val_loader, class_names, device=device)
plot_confusion_matrix(y_true, y_pred, class_names)
